# 🎬 Wan2.1 — Google Colab

Este notebook prepara o repositório **Wan2.1** para rodar no Google Colab.

> **Objetivo:** clonar o código, instalar as dependências, baixar os pesos do modelo e
> executar a geração de vídeo (texto → vídeo) com o modelo **T2V-1.3B**, que é o mais
> leve e funciona bem na GPU grátis do Colab.

## Como abrir este notebook
- Faça o upload deste `.ipynb` para:
  `https://colab.research.google.com`
- Ou abra direto do GitHub:
  `https://colab.research.google.com/github/anonyby777-lgtm/wan2.1/blob/arena/01a05b3f-wan2-1/Wan2.1_Colab.ipynb`

## Antes de começar
1. No menu `Runtime` → `Change runtime type`, escolha **GPU** (T4 é suficiente para o 1.3B).
2. Rode as células na ordem.
3. Se o Colab desconectar durante o download do modelo, monte o Drive e salve os pesos lá.

## Cuidados com o espaço em disco
- **T2V-1.3B:** ~2,5 GB → ok no Colab gratuito.
- **14B:** ~28+ GB → precisa de disco grande (e, na prática, uma assinatura do Colab).
- Por isso este notebook baixa **apenas o 1.3B** por padrão. As células dos modelos maiores
  ficam comentadas.

---


In [ ]:
# 0) Verifique a GPU disponível
!nvidia-smi

import torch
print("CUDA disponível:", torch.cuda.is_available())
print("Versão CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memória total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Passo 1 — Montar o Google Drive (opcional)

Se você quiser manter modelos e vídeos no Drive (útil quando o Colab reinicia), descomente a
linha abaixo. Caso contrário, pode ignorar.


In [ ]:
# 1) (Opcional) Montar o Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


## Passo 2 — Clonar o repositório

Clona **este** repositório (`anonyby777-lgtm/wan2.1`) e entra na pasta.


In [ ]:
# 2) Clonar o repositório
!rm -rf /content/Wan2.1
!git clone https://github.com/anonyby777-lgtm/wan2.1.git /content/Wan2.1
%cd /content/Wan2.1
print("OK, repositório clonado.")


## Passo 3 — Instalar dependências

O `requirements.txt` original inclui `flash_attn`, que quase sempre **falha** para compilar
no Colab. Como o próprio `wan/modules/attention.py` já tem fallback para
`torch.nn.functional.scaled_dot_product_attention` quando o flash-attn não existe, você não
precisa dele aqui.

Por segurança, instalamos as dependências sem `flash_attn` (e sem instalar o pacote via
`pip install -e .`, que tentaria puxar `flash_attn` de novo).


In [ ]:
# 3) Instalar dependências
import sys, subprocess

deps = [
    "torch>=2.4.0",
    "torchvision>=0.19.0",
    "opencv-python>=4.9.0.80",
    "diffusers>=0.31.0",
    "transformers>=4.49.0",
    "tokenizers>=0.20.3",
    "accelerate>=1.1.1",
    "tqdm",
    "imageio",
    "easydict",
    "ftfy",
    "dashscope",
    "imageio-ffmpeg",
    "gradio>=5.0.0",
    "numpy>=1.23.5,<2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[cli]"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "modelscope"] + deps)
print("Dependências instaladas.")
print("Torch:", __import__('torch').__version__)


## Passo 4 — Baixar os pesos do modelo

Vamos baixar **Wan2.1-T2V-1.3B** (~2,5 GB). É a opção recomendada para Colab gratuito.

Para outros modelos, descomente uma das opções no fim da célula (a lista precisa de muito
mais disco/VRAM).


In [ ]:
# 4) Baixar o modelo T2V-1.3B (recomendado para Colab gratuito)
%cd /content/Wan2.1
!huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./Wan2.1-T2V-1.3B
print("Modelo T2V-1.3B baixado.")


In [ ]:
# 4b) (Opcional) Baixar outros modelos
# Descomente apenas o que precisar. Cada um é grande.

# Texto -> Video 14B (~28 GB)
# !huggingface-cli download Wan-AI/Wan2.1-T2V-14B --local-dir ./Wan2.1-T2V-14B

# Imagem -> Video 14B 480P
# !huggingface-cli download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./Wan2.1-I2V-14B-480P

# Imagem -> Video 14B 720P
# !huggingface-cli download Wan-AI/Wan2.1-I2V-14B-720P --local-dir ./Wan2.1-I2V-14B-720P

# Primeiro/último frame -> Vídeo 14B 720P
# !huggingface-cli download Wan-AI/Wan2.1-FLF2V-14B-720P --local-dir ./Wan2.1-FLF2V-14B-720P

# VACE 1.3B (edição/geral)
# !huggingface-cli download Wan-AI/Wan2.1-VACE-1.3B --local-dir ./Wan2.1-VACE-1.3B

# VACE 14B
# !huggingface-cli download Wan-AI/Wan2.1-VACE-14B --local-dir ./Wan2.1-VACE-14B


## Passo 5 — Gerar vídeo (texto → vídeo)

Comando equivalente ao exemplo oficial do README, usando o 1.3B em 480P.
A primeira execução pode demorar alguns minutos enquanto os pesos são carregados.


In [ ]:
# 5) Gerar um vídeo de exemplo
%cd /content/Wan2.1

import subprocess, sys
cmd = [
    sys.executable, "generate.py",
    "--task", "t2v-1.3B",
    "--size", "832*480",
    "--ckpt_dir", "./Wan2.1-T2V-1.3B",
    "--frame_num", "81",
    "--sample_steps", "50",
    "--sample_guide_scale", "6.0",
    "--base_seed", "42",
    "--prompt", "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage.",
]
subprocess.check_call(cmd)
print("Geração concluída.")


## Passo 6 — Visualizar o vídeo gerado

O vídeo `.mp4` é salvo na raiz de `/content/Wan2.1`.


In [ ]:
# 6) Exibir o vídeo gerado
from IPython.display import Video, display
import glob, os

videos = glob.glob("/content/Wan2.1/*.mp4")
if not videos:
    print("Nenhum .mp4 encontrado. Rode a célula 5 primeiro.")
else:
    latest = max(videos, key=os.path.getmtime)
    print("Vídeo:", latest)
    print("Tamanho (MB):", round(os.path.getsize(latest) / (1024**2), 2))
    display(Video(latest, embed=True, width=640))


## Passo 7 — Baixar o vídeo para o seu computador (opcional)


In [ ]:
# 7) Baixar o último vídeo gerado para o seu computador
from google.colab import files
import glob, os

videos = glob.glob("/content/Wan2.1/*.mp4")
if videos:
    latest = max(videos, key=os.path.getmtime)
    files.download(latest)
else:
    print("Nenhum .mp4 encontrado.")


## Exemplos adicionais

Abaixo alguns comandos úteis. Ajuste `--task` e `--ckpt_dir` conforme o modelo baixado.

### Texto → Imagem
```bash
python generate.py --task t2i-14B --size "1024*1024" \
  --ckpt_dir ./Wan2.1-T2V-14B \
  --prompt "Uma pequena casa no campo ao pôr do sol"
```

### Imagem → Vídeo (I2V-14B-480P)
```bash
python generate.py --task i2v-14B --size "480*832" \
  --ckpt_dir ./Wan2.1-I2V-14B-480P \
  --image examples/i2v_input.JPG \
  --prompt "A white cat wearing sunglasses sits on a surfboard."
```

### Primeiro + último frame → Vídeo (FLF2V-14B-720P)
```bash
python generate.py --task flf2v-14B --size "720*1280" \
  --ckpt_dir ./Wan2.1-FLF2V-14B-720P \
  --first_frame examples/flf2v_input_first_frame.png \
  --last_frame examples/flf2v_input_last_frame.png \
  --prompt "CG动画风格，一只蓝色的小鸟从地面起飞. 近景, 仰视视角."
```

### VACE (edição geral)
```bash
python generate.py --task vace-1.3B --size "480*832" \
  --ckpt_dir ./Wan2.1-VACE-1.3B \
  --src_ref_images examples/girl.png,examples/snake.png \
  --prompt "Um gato brincando em um jardim ensolarado"
```


## 🧠 Notas importantes

- **Colab gratuito:** use **T2V-1.3B** em 480P (832×480 ou 480×832). A 14B é muito pesada
  para a sessão gratuita.
- **Flash-attn:** não tentamos instalar; o código usa `scaled_dot_product_attention`
  automaticamente.
- **Reinício do Colab:** os downloads feitos em `/content` são apagados quando a sessão
  reinicia. Para preservar modelos em execuções longas, monte o Drive e use um caminho
  dentro de `/content/drive`.
- **Memória:** se ocorrer OOM, reduza `--size`, `--frame_num` ou `--sample_steps`.
